# Check GPU Device

In [1]:
# Use the shell escape (!) to run the script on Colab's cloud filesystem
!python -c "import torch; print('CUDA Available:', torch.cuda.is_available()); print('GPU Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')"


CUDA Available: True
GPU Device: NVIDIA L4


# Mount Google Drive and Git Clone the repo

In [2]:
import os
from google.colab import drive

# 1. Mount Google Drive to access your processed dataset
print("Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Define repository details (Using public HTTPS url)
repo_name = "SME_Credit_Risk"
repo_url = f"https://github.com/mirkosimunovic/{repo_name}.git"

# 3. Clone the public repo (or pull if it already exists)
if not os.path.exists(f"/content/{repo_name}"):
    print(f"\nCloning public repository: {repo_url}...")
    !git clone {repo_url}
else:
    print(f"\nRepository {repo_name} already exists. Pulling latest code changes...")
    %cd /content/{repo_name}
    !git pull
    %cd /content

# 4. Create the target directory inside the cloned repo
!mkdir -p /content/{repo_name}/data/raw

# 5. Copy the cleaned SME dataset from Google Drive
# NOTE: If you saved the CSV inside a specific folder in Google Drive, 
# adjust the source path below (e.g., "/content/drive/MyDrive/YourFolder/processed_sme_final.csv")
drive_source_path = "/content/drive/MyDrive/xAI_Banking_Paper/data/SBAnational.csv"
colab_target_path = f"/content/{repo_name}/data/raw/SBAnational.csv"

if os.path.exists(drive_source_path):
    !cp "{drive_source_path}" "{colab_target_path}"
    print("\n✓ Cleaned SBA dataset successfully copied from Google Drive to the cloned project")
else:
    print(f"\n⚠️ WARNING: Could not find your dataset at: {drive_source_path}")
    print("Please check your file path inside your Google Drive side panel and update 'drive_source_path'.")

# 6. Change active directory to your repository root
%cd /content/{repo_name}
print(f"\nActive directory set to: {os.getcwd()}")

Mounted at /content/drive

Cloning public repository: https://github.com/mirkosimunovic/SME_Credit_Risk.git...
Cloning into 'SME_Credit_Risk'...
remote: Enumerating objects: 216, done.
remote: Counting objects: 100% (216/216), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 216 (delta 88), reused 180 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (216/216), 18.82 MiB | 20.28 MiB/s, done.
Resolving deltas: 100% (88/88), done.

✓ Cleaned SBA dataset successfully copied from Google Drive to the cloned project
/content/SME_Credit_Risk

Active directory set to: /content/SME_Credit_Risk


# Sync the Google Drive files into my Colab session repo

In [4]:
import shutil
from pathlib import Path

drive_root = Path("/content/drive/MyDrive/xAI_Banking_Paper/SME_Credit_Risk")
colab_root = Path("/content/SME_Credit_Risk")

files = [
    "data/processed/X_train.csv",
    "data/processed/y_train.csv",
    "data/processed/X_oot.csv",
    "data/processed/y_oot.csv",
    "models/artifacts/imputer.joblib",
    "models/artifacts/scaler.joblib",
    "models/artifacts/xgboost_best.json",
    "models/artifacts/lightgbm_best.txt",
    "models/artifacts/catboost_best.bin"

]

if not drive_root.exists():
    raise FileNotFoundError(
        f"Drive folder not found: {drive_root}\n"
        "Mount Drive and check the folder name."
    )

for rel in files:
    source = drive_root / rel
    dest = colab_root / rel
    if not source.exists():
        print(f"SKIP (not on Drive): {rel}")
        continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, dest)
    mb = source.stat().st_size / 1e6
    print(f"copied {rel}  ({mb:.1f} MB)")

print("\nDone. Files are in the Colab repo.")

copied data/processed/X_train.csv  (68.2 MB)
copied data/processed/y_train.csv  (1.5 MB)
copied data/processed/X_oot.csv  (12.1 MB)
copied data/processed/y_oot.csv  (0.3 MB)
copied models/artifacts/imputer.joblib  (134.2 MB)
copied models/artifacts/scaler.joblib  (0.0 MB)
copied models/artifacts/xgboost_best.json  (2.1 MB)
copied models/artifacts/lightgbm_best.txt  (1.1 MB)
copied models/artifacts/catboost_best.bin  (0.3 MB)

Done. Files are in the Colab repo.


# Git Pull to sync

In [3]:
!git pull


Already up to date.


# Install requirements

In [4]:

# 1. Install heavy, CUDA-dependent system libraries first
!pip install "fknni[rapids12]" --extra-index-url=https://pypi.nvidia.com
!pip install faiss-gpu-cu12

# 2. Silently install the rest of our standard project requirements
!pip install -q -r requirements.txt


Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 812.0 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 243.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.5/114.5 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.2/51.2 kB 206.3 kB/s eta 0:00:00:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 GB 3.9 MB/s eta 0:00:0000:0100:06m
INFO: pip is looking at multiple versions of cugraph-cu12 to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 8.1 MB/s eta 0:00:00a 0:00:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 4.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 527.9 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 85.0 MB/s eta 0:00:00
     ━━━━━

# Check project tree

In [8]:
!find . -not -path './.git/*' -not -path './__pycache__/*' | sed -e 's|[^/]*/|│   |g' -e 's|│   \([^│]\)|├── \1|'

.
├── models
│   ├── artifacts
│   │   ├── .gitkeep
│   │   ├── xgboost_best_v3.json
│   │   ├── catboost_best_v3.bin
│   │   ├── scaler_v3.joblib
│   │   ├── lightgbm_best_v3.txt
│   │   ├── imputer_v3.joblib
├── .gitignore
├── scripts
│   ├── trainer_v2.py
│   ├── preprocess.py
│   ├── explainer_v2.py
│   ├── causal_inference_v3.py
│   ├── trainer.py
│   ├── explainer_v3.py
│   ├── explainer.py
│   ├── preprocess_v2.py
│   ├── causal_inference.py
│   ├── trainer_v3.py
│   ├── preprocess_v3.py
│   ├── causal_inference_v2.py
│   ├── policy_simulator_v3.py
├── notebooks
│   ├── run_commands.ipynb
│   ├── SME_week1.ipynb
├── .cursorrules
├── data
│   ├── raw
│   │   ├── SBAnational.csv
│   ├── processed
│   │   ├── .gitkeep
│   │   ├── y_train_v3.csv
│   │   ├── y_oot_v3.csv
│   │   ├── X_train_v3.csv
│   │   ├── X_oot_v3.csv
├── requirements.txt
├── .git
├── outputs
│   ├── figures
│   │   ├── shap_summary.png
│   │   ├── roc_curves_comparison.png
│   │   ├── shap_importance.png
│   │  

# Run pipeline scripts

In [16]:
!python scripts/preprocess.py

SBA SME preprocessing — chronological, leak-proof

[1] Load data/raw/SBAnational.csv
  Raw shape: (899164, 27)
  Dropped 1,997 rows with missing / unmapped MIS_Status.
  Paid in Full (0)=739,609 | Default (1)=157,558

[1b] Parse GrAppv and SBA_Appv currency strings -> float
  GrAppv: non-null=897,167  min=1000.0  max=5472000.0
  SBA_Appv: non-null=897,167  min=500.0  max=5472000.0

[2] Parse ApprovalDate, sort oldest -> newest, chronological split
  Rolled 4 two-digit years back by 100 years.
  Sorted span: 1966-05-18 -> 2014-06-25
  Train (oldest 85%): 1966-05-18 -> 2007-03-21  n=762,591
  OOT   (newest 15%): 2007-03-21 -> 2014-06-25  n=134,576
  Default rate  train=0.1532  OOT=0.3029

[3] NAICS_Sector + training-only target encoding (State, sector)
  State keys scored from train: 52
  Safest states (lowest train default rate):
State
MT    0.062095
WY    0.063191
VT    0.069453
SD    0.073834
ND    0.079167
  Riskiest states (highest train default rate):
State
GA    0.198766
IL    0.2

In [17]:
!python scripts/trainer.py

SME Credit Risk — two-step trainer + OOT evaluation
Project root: /content/SME_Credit_Risk

Creating output directories if missing:
  [ok] outputs/figures
  [ok] outputs/results
  [ok] models/artifacts

Loading chronological splits (NaNs still present):
  Loaded train: X=(762591, 22)  defaults=116,798  rate=0.153159
  Loaded oot: X=(134576, 22)  defaults=40,760  rate=0.302877

STEP 1  Stratified 5-fold CV on X_train / y_train
Imputer: FastKNNImputer.fit_transform on the train fold, then on stacked val.
         (Library has no fit/transform; val neighbors come from imputed train.)
Scaler:  StandardScaler on continuous columns (nunique > 10) — train fold only.
Imbalance: scale_pos_weight = n_neg / n_pos on the training fold.
OOT is held out of this entire loop.

Fold 1/5  n_train=610,072  n_val=152,519  scale_pos_weight=5.53
    FastKNNImputer.fit_transform on training slice (610,072 rows) ...
    FastKNNImputer.fit_transform on stacked reference+apply (762,591 rows) ...
    Continuous 

In [18]:
!python scripts/explainer.py

Week 3  SHAP explainability + Kendall's W stability (XGBoost champion)

[2] Loading frozen imputer / scaler and chronological splits
  Loading models/artifacts/imputer.joblib ...
  Imputer reference shape=(762591, 22)  encoded columns=22
  Loading models/artifacts/scaler.joblib ...
  Loading OOT features / targets ...
  Loaded oot: X=(134576, 22)  default rate=0.302877
  Transforming OOT (impute against frozen train reference, then scale) ...
    FastKNNImputer.fit_transform on stacked reference+apply (897,167 rows) ...
    Scaling 12 continuous columns: ['State', 'NAICS', 'Term', 'NoEmp', 'CreateJob', 'RetainedJob', 'RevLineCr', 'GrAppv', 'NAICS_Sector', 'Log_GrAppv', 'Guarantee_Ratio', 'Term_Years']
  Loading training features / targets ...
  Loaded train: X=(762591, 22)  default rate=0.153159
  Using imputer.reference_ as imputed X_train (champion training matrix) ...
    Scaling 12 continuous columns: ['State', 'NAICS', 'Term', 'NoEmp', 'CreateJob', 'RetainedJob', 'RevLineCr', 'GrA

In [8]:
!python scripts/causal_inference.py

Causal validation — GCM falsification, LinearDML, refutation suite
Creating output directories:
  [ok] outputs/figures
  [ok] outputs/results

Loading chronological training split ...
  Loaded train: (762591, 23)  default rate=0.153159
  Observed confounders used: ['Log_GrAppv', 'NoEmp', 'NewExist_Clean', 'State_Points', 'NAICS_Sector_Points', 'UrbanRural']
  Complete-case rows for causal DAG: 761,716 / 762,591

PHASE 1  GCM permutation falsification of the proposed DAG
  GCM sample n=10,000  default rate=0.1533
  DAG nodes=9  edges=21
  Running falsify_graph (n_permutations=100, GCM independence) ...
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/li

# Pipeline V2

In [6]:
!python scripts/preprocess_v2.py


SBA SME preprocessing v2 — 60-month window + chronological 85/15 OOT

[1] Load data/raw/SBAnational.csv
  Raw shape: (899164, 27)
  Dropped 1,997 rows with missing / unmapped MIS_Status.
  Paid in Full (0)=739,609 | Default (1)=157,558

[1b] Parse GrAppv and SBA_Appv currency strings -> float
  GrAppv: non-null=897,167  min=1000.0  max=5472000.0
  SBA_Appv: non-null=897,167  min=500.0  max=5472000.0

[1c] Enforce 60-month performance window via DisbursementDate
  Rolled 5 DisbursementDate two-digit years back by 100 years.
  Dropping 2,175 rows with unparseable DisbursementDate.
  Observation end=2014-12-31  censor cutoff=2009-12-31  kept 858,516 / 894,992 loans.

[2] Parse ApprovalDate, sort oldest -> newest, chronological 85/15 split
  Rolled 4 ApprovalDate two-digit years back by 100 years.
  Sorted span: 1966-05-18 -> 2009-12-31
  Train (oldest 85%): 1966-05-18 -> 2006-10-16  n=729,738
  OOT   (newest 15%): 2006-10-16 -> 2009-12-31  n=128,778
  Default rate  train=0.1422  OOT=0.390

In [7]:
!python scripts/trainer_v2.py


SME Credit Risk v2 — tuned trainer + OOT bootstrap CIs
Project root: /content/SME_Credit_Risk  GPU=True (xgb device=cuda)

Creating output directories if missing:
  [ok] outputs/figures
  [ok] outputs/results
  [ok] models/artifacts

Loading chronological splits (NaNs still present):
  Loaded train: X=(729738, 15)  defaults=103,759  rate=0.142187
  Loaded oot: X=(128778, 15)  defaults=50,270  rate=0.390362

STEP 1  Stratified 5-fold CV on X_train / y_train
Imputer: FastKNNImputer.fit_transform on the train fold, then on stacked val.
         (Library has no fit/transform; val neighbors come from imputed train.)
Scaler:  StandardScaler on continuous columns (nunique > 10) — train fold only.
Imbalance: scale_pos_weight = n_neg / n_pos on the training fold.
OOT is held out of this entire loop.

Fold 1/5  n_train=583,790  n_val=145,948  scale_pos_weight=6.03
    FastKNNImputer.fit_transform on training slice (583,790 rows) ...
    FastKNNImputer.fit_transform on stacked reference+apply (72

In [8]:
!python scripts/explainer_v2.py


Week 3  SHAP explainability + Kendall's W stability (XGBoost champion)

[2] Loading frozen imputer / scaler and chronological splits
  Loaded tuned XGBoost params from outputs/results/best_hyperparams_v2.json
  XGB_BASELINE = {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.7, 'min_child_weight': 5, 'reg_lambda': 5.0, 'tree_method': 'hist', 'eval_metric': 'logloss', 'n_jobs': -1, 'verbosity': 0}
  Loading models/artifacts/imputer_v2.joblib ...
  Imputer reference shape=(729738, 15)  encoded columns=15
  Loading models/artifacts/scaler_v2.joblib ...
  Loading OOT features / targets ...
  Loaded oot: X=(128778, 15)  default rate=0.390362
  Transforming OOT (impute against frozen train reference, then scale) ...
    FastKNNImputer.fit_transform on stacked reference+apply (858,516 rows) ...
    Scaling 5 continuous columns: ['NoEmp', 'RevLineCr', 'Log_GrAppv', 'Guarantee_Ratio', 'Term_Years']
  Loading training features / targets ...
  Lo

In [9]:
!python scripts/causal_inference_v2.py




Causal validation — GCM falsification, LinearDML, refutation suite
Creating output directories:
  [ok] outputs/figures
  [ok] outputs/results

Loading chronological training split ...
  Loaded train: (729738, 16)  default rate=0.142187
  Observed confounders used: ['Log_GrAppv', 'NoEmp', 'NewExist_Clean', 'State_Points', 'NAICS_Sector_Points', 'UrbanRural']
  Complete-case rows for causal DAG: 728,872 / 729,738
  Tuning LGBMRegressor nuisance model on 30,000 rows ...
  Nuisance best params: {'num_leaves': 31, 'n_estimators': 150, 'max_depth': 8, 'learning_rate': 0.03}

PHASE 1  GCM permutation falsification of the proposed DAG
  GCM sample n=10,000  default rate=0.1423
  DAG nodes=9  edges=21
  Running falsify_graph (n_permutations=100, GCM independence) ...
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sk

# Pipeline V3

In [5]:
!python scripts/preprocess_v3.py

SBA SME preprocessing v3 — 60-month window + chronological 85/15 OOT

[1] Load data/raw/SBAnational.csv
  Raw shape: (899164, 27)
  Dropped 1,997 rows with missing / unmapped MIS_Status.
  Paid in Full (0)=739,609 | Default (1)=157,558

[1b] Parse GrAppv and SBA_Appv currency strings -> float
  GrAppv: non-null=897,167  min=1000.0  max=5472000.0
  SBA_Appv: non-null=897,167  min=500.0  max=5472000.0

[1c] Enforce 60-month performance window via DisbursementDate
  Rolled 5 DisbursementDate two-digit years back by 100 years.
  Dropping 2,175 rows with unparseable DisbursementDate.
  Observation end=2014-12-31  censor cutoff=2009-12-31  retained 858,516  excluded post-2010 36,476

  Distributional shift test (retained pre-2010 vs excluded post-2010)
  Post-2010 loans remain excluded to protect the 60-month default window.
  feature                 KS       p-value       PSI                status  drift
  ------------------------------------------------------------------------------
  GrAp

In [6]:
!python scripts/trainer_v3.py


SME Credit Risk v3 — financial weights + balanced_accuracy tuning
Project root: /content/SME_Credit_Risk  GPU=True (xgb device=cuda)

Creating output directories if missing:
  [ok] outputs/figures
  [ok] outputs/results
  [ok] models/artifacts

Loading chronological splits (NaNs still present):
  Loaded train: X=(729738, 15)  defaults=103,759  rate=0.142187
  Loaded oot: X=(128778, 15)  defaults=50,270  rate=0.390362

STEP 1  Stratified 5-fold CV on X_train / y_train
Imputer: FastKNNImputer.fit_transform on the train fold, then on stacked val.
         (Library has no fit/transform; val neighbors come from imputed train.)
Scaler:  StandardScaler on continuous columns (nunique > 10) — train fold only.
Weights: financial sample weights on unscaled rows (ECL vs interest opportunity).
Tuning:  RandomizedSearchCV scoring=balanced_accuracy. No scale_pos_weight.
OOT is held out of this entire loop.

Fold 1/5  n_train=583,790  n_val=145,948  weight mean=1.000  weight max=31.94
    FastKNNImput

In [9]:

!python scripts/explainer_v3.py


Week 3 v3  SHAP + Kendall's W (profit-sensitive XGBoost)

[2] Loading frozen imputer / scaler and chronological splits
  Loaded tuned XGBoost params from outputs/results/best_hyperparams_v3.json
  XGB_BASELINE = {'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.1, 'subsample': 1.0, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'reg_lambda': 1.0, 'tree_method': 'hist', 'eval_metric': 'logloss', 'n_jobs': -1, 'verbosity': 0}
  Loading models/artifacts/imputer_v3.joblib ...
  Imputer reference shape=(729738, 15)  encoded columns=15
  Loading models/artifacts/scaler_v3.joblib ...
  Loading OOT features / targets ...
  Loaded oot: X=(128778, 15)  default rate=0.390362
  Transforming OOT (impute against frozen train reference, then scale) ...
    FastKNNImputer.fit_transform on stacked reference+apply (858,516 rows) ...
    Scaling 5 continuous columns: ['NoEmp', 'RevLineCr', 'Log_GrAppv', 'Guarantee_Ratio', 'Term_Years']
  Loading training features / targets ...
  Loaded train: X=(

In [9]:

!python scripts/causal_inference_v3.py


Causal validation v3 — GCM falsification, LinearDML, refutation suite
Creating output directories:
  [ok] outputs/figures
  [ok] outputs/results

Loading chronological training split ...
  Loaded train: (729738, 16)  default rate=0.142187
  Observed confounders used: ['Log_GrAppv', 'NoEmp', 'NewExist_Clean', 'State_Points', 'NAICS_Sector_Points', 'UrbanRural']
  Complete-case rows for causal DAG: 728,872 / 729,738
  Tuning LGBMRegressor nuisance model on 30,000 rows ...
  Nuisance best params: {'num_leaves': 31, 'n_estimators': 150, 'max_depth': 8, 'learning_rate': 0.03}

PHASE 1  GCM permutation falsification of the proposed DAG
  GCM sample n=10,000  default rate=0.1423
  DAG nodes=9  edges=21
  Running falsify_graph (n_permutations=100, GCM independence) ...
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages

: 

In [7]:

!python scripts/policy_simulator_v3.py

Policy simulator v3 — XGBoost vs LinearDML term extension

[1] Load v3 splits and frozen XGBoost pipeline
  Loaded train: X=(729738, 15)  default rate=0.142187
  Loaded oot: X=(128778, 15)  default rate=0.390362
  Loading models/artifacts/imputer_v3.joblib

[2] Fit LinearDML Model A on complete-case X_train_v3
  Complete-case train for LinearDML: 728,872 / 729,738
  Tuning LGBMRegressor nuisance model on 30,000 rows ...
  Nuisance best params: {'num_leaves': 31, 'n_estimators': 150, 'max_depth': 8, 'learning_rate': 0.03}
  Fitting LinearDML (Model A: Term_Years; no GCM) ...
  LinearDML ATE (Term_Years -> default) = -0.020647

[3] Baseline XGBoost probabilities on OOT
    FastKNNImputer.fit_transform on stacked reference+apply (858,516 rows) ...
  Approved=86,942 / 128,778  Baseline_Portfolio_Value=10,676,487,646

[4] Marginal rejections: 0.50 <= P <= 0.65  n=7,288

[5] Naive ML policy — XGBoost on Term_Years + 5 (targeted rows only)
    FastKNNImputer.fit_transform on stacked reference

# Sync the Colab session v3 outputs to my Google Drive

In [9]:
import shutil
from pathlib import Path

src = Path("/content/SME_Credit_Risk")
dst = Path("/content/drive/MyDrive/xAI_Banking_Paper/SME_Credit_Risk")

# files = [
#     # preprocess.py (chronological splits — re-run on Colab)
#     "data/processed/X_train.csv",
#     "data/processed/y_train.csv",
#     "data/processed/X_oot.csv",
#     "data/processed/y_oot.csv",
#     # trainer.py
#     "models/artifacts/imputer.joblib",
#     "models/artifacts/scaler.joblib",
#     "models/artifacts/xgboost_best.json",
#     "models/artifacts/lightgbm_best.txt",
#     "models/artifacts/catboost_best.bin",
#     "outputs/results/metrics_benchmark.csv",
#     "outputs/figures/roc_curves_comparison.png",
#     # explainer.py
#     "outputs/figures/shap_summary.png",
#     "outputs/figures/shap_importance.png",
#     "outputs/results/stability_report.json"
# ]

files = [
    # preprocess_v3.py
    "data/processed/X_train_v3.csv",
    "data/processed/y_train_v3.csv",
    "data/processed/X_oot_v3.csv",
    "data/processed/y_oot_v3.csv",
    "outputs/results/distributional_shift_report_v3.json",
    # trainer_v3.py
    "models/artifacts/imputer_v3.joblib",
    "models/artifacts/scaler_v3.joblib",
    "models/artifacts/xgboost_best_v3.json",
    "models/artifacts/lightgbm_best_v3.txt",
    "models/artifacts/catboost_best_v3.bin",
    "outputs/results/metrics_benchmark_v3.csv",
    "outputs/results/best_hyperparams_v3.json",
    "outputs/figures/roc_curves_comparison_v3.png",
    # explainer_v3.py
    "outputs/figures/shap_summary_v3.png",
    "outputs/figures/shap_importance_v3.png",
    "outputs/results/stability_report_v3.json",
    # causal_inference_v3.py
    "outputs/results/causal_stability_report_v3.json",
    "outputs/figures/gcm_falsify_histogram_v3.png",
    # policy_simulator_v3.py
    "outputs/results/policy_simulation_v3.json",
]

if not dst.exists():
    raise FileNotFoundError(
        f"Drive folder not found: {dst}\n"
        "Re-run the drive.mount cell, then check the folder name."
    )

for rel in files:
    source = src / rel
    dest = dst / rel
    if not source.exists():
        print(f"SKIP (missing in Colab): {rel}")
        continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, dest)
    mb = source.stat().st_size / 1e6
    print(f"copied {rel}  ({mb:.1f} MB)")

print("\nDone. Confirm on Drive, then download that folder into your local SME_Credit_Risk repo.")

copied outputs/results/causal_stability_report.json  (0.0 MB)
copied outputs/figures/gcm_falsify_histogram.png  (0.1 MB)

Done. Confirm on Drive, then download that folder into your local SME_Credit_Risk repo.


In [ ]:
!ls